# 🚀 Notebook do Professor (Demo) — Aula 12: Router chains e o conceito de grafo de estado

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 12/14 — Módulo 4: LangGraph e Encerramento**  
**⏱️ 1h40min**  
**🔀 Router Chain · StateGraph mental · LangGraph motivação**  
**🔁 Andaime 50%**  

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções completas dos exercícios de fixação.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — Classificador de intenção — o coração do Router

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal

llm = ChatOllama(model="gpt-oss:120b", temperature=0)

# Schema Pydantic para a saída do classificador
class Rota(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]
    # Literal garante que o LLM retorne APENAS uma dessas 3 strings
    # Se retornar algo diferente → ValidationError → fallback para "conversa"

prompt_classificador = ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input do usuário em exatamente um destino:
- rag: perguntas sobre documentos do domínio, manuais, contratos, regulamentos
- calculadora: cálculos matemáticos, conversões numéricas, porcentagens
- conversa: saudações, perguntas gerais, agradecimentos, qualquer outra coisa

Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
])

# Chain do classificador — with_structured_output garante Pydantic válido
chain_classificador = prompt_classificador | llm.with_structured_output(Rota)

# Testar o classificador isoladamente
for inp in ["Qual a cláusula 5 do contrato?", "Quanto é 450 × 0.88?", "Oi tudo bem?"]:
    rota = chain_classificador.invoke({"input": inp})
    print(f"'{inp[:30]}...' → {rota.destino}")
# 'Qual a cláusula 5 do contrato?...' → rag
# 'Quanto é 450 × 0.88?...'          → calculadora
# 'Oi tudo bem?...'                  → conversa

### Slide 08 — Handlers — uma chain por rota

In [ ]:
from langchain_core.runnables import RunnableLambda

# Handler 1 — RAG (reutiliza o retriever e prompt do CKP02)
prompt_rag = ChatPromptTemplate.from_template(
    "Use o contexto abaixo para responder a pergunta. Se não souber, diga que não sabe.\n"
    "{contexto}\n\nPergunta: {input}"
)
handler_rag = (
    {"contexto": retriever | RunnableLambda(lambda docs: "\n\n".join(d.page_content for d in docs)),
     "input": lambda x: x["input"]}
    | prompt_rag | llm | StrOutputParser()
)

# Handler 2 — calculadora (sem LLM para a matemática)
def _calcular(dados: dict) -> str:
    # Primeiro extrai a expressão Python via LLM, depois calcula
    expressao = (ChatPromptTemplate.from_template(
        "Extraia apenas a expressão Python do cálculo pedido. Retorne somente a expressão, sem texto: {input}"
    ) | llm | StrOutputParser()).invoke(dados)
    try:
        resultado = eval(expressao.strip(), {"__builtins__": {}}, {})
        return f"Resultado: {resultado} (expressão: {expressao.strip()})"
    except:
        return "Não foi possível calcular. Tente escrever a expressão de outra forma."
handler_calculadora = RunnableLambda(_calcular)

# Handler 3 — conversa genérica
handler_conversa = (
    ChatPromptTemplate.from_messages([
        ("system", "Você é um assistente simpático do domínio do grupo. Responda de forma amigável e concisa."),
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

# Mapa de handlers por destino
HANDLERS = {"rag": handler_rag, "calculadora": handler_calculadora, "conversa": handler_conversa}

### Slide 09 — Montando o Router — classificar → despachar

In [ ]:
from langchain_core.runnables import RunnableLambda

def rotear(dados: dict) -> str:
    """Classifica a intenção e despacha para o handler correto."""
    inp  = dados["input"]
    rota = chain_classificador.invoke({"input": inp})

    print(f"[ROUTER] '{inp[:40]}' → {rota.destino}")

    # Despachar para o handler correspondente
    handler = HANDLERS.get(rota.destino, HANDLERS["conversa"])  # fallback: conversa
    return handler.invoke({"input": inp})

# Router chain final — RunnableLambda torna a função uma Runnable LCEL
router_chain = RunnableLambda(rotear)

# Testar o sistema completo
testes = [
    "Qual o prazo de garantia no contrato?",   # → rag
    "Quanto é 1.250 com 15% de desconto?",     # → calculadora
    "Você pode me ajudar com algo hoje?",      # → conversa
    "Qual a multa por rescisão antecipada?",    # → rag
]
for t in testes:
    print(f"\n{'='*50}\nInput: {t}\nOutput: {router_chain.invoke({'input': t})}")

### Slide 17 — with_structured_output() — structured output revisitado

In [ ]:
# LLM pode retornar qualquer texto
"rag"          # ✅ ok
"RAG"          # ❌ KeyError no dict
"use o rag"    # ❌ KeyError no dict
"Vou usar rag" # ❌ quebra em produção

### Slide 17 — with_structured_output() — structured output revisitado

In [ ]:
# Literal garante exatamente 1 de N
class Rota(BaseModel):
    destino: Literal["rag","calc","chat"]
# O LLM é forçado (via JSON schema)
# a retornar APENAS um dos 3 valores
# "RAG" → ValidationError → tratável

### Slide 20 — Python novo desta aula

In [ ]:
# 1. Literal — tipo que aceita apenas valores específicos
from typing import Literal

def fn(cor: Literal["vermelho", "azul", "verde"]): ...
# fn("amarelo") → erro de type checking (mas não RuntimeError em Python puro)
# com Pydantic: Literal em BaseModel → ValidationError em runtime

# 2. with_structured_output(PydanticModel) — reforçar Literal em runtime
from pydantic import BaseModel

class Rota(BaseModel):
    destino: Literal["a", "b"]

chain = prompt | llm.with_structured_output(Rota)
result = chain.invoke({"input": "..."})
result.destino  # → sempre "a" ou "b" — nunca outro valor

# 3. dict.get(key, default) — fallback seguro
d = {"a": 1, "b": 2}
d.get("c", "default")  # → "default" (sem KeyError)
d["c"]                   # → KeyError (quebra em produção)

# 4. RunnableLambda — transformar qualquer função em Runnable LCEL
from langchain_core.runnables import RunnableLambda

def minha_fn(dados: dict) -> str: return dados["x"].upper()
runnable = RunnableLambda(minha_fn)
runnable.invoke({"x": "hello"})  # → "HELLO"
# RunnableLambda é o "adaptador" que coloca qualquer função no pipe LCEL

# 5. Lambda em dicts de chain — LCEL fan-out
chain = {
    "contexto": retriever | RunnableLambda(lambda docs: "\n".join(d.page_content for d in docs)),
    "input": lambda x: x["input"],
} | prompt | llm  # dict com lambdas = fan-out LCEL (executa em paralelo)

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Classificador de intenção com Literal

**O que a solução demonstra:** o andaime do Exercício 1 do aluno já preenchido — `Literal` com os 3 destinos, descrição de cada rota no prompt e `with_structured_output` — rodando os 3 testes do andaime.

**Pontos a destacar em sala:**
- Rodar a versão com lacunas ao lado (no notebook do aluno) e esta: a única diferença são os valores nas lacunas — `Literal`, descrições e o método do `llm`.
- Se um teste sair na rota errada, a correção é na descrição da rota no prompt, não no schema: o `Literal` garante o formato, o texto garante a intenção.


In [ ]:
# Solução — classificador de intenção com Literal (Exercício 1)
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class RotaEx1(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

prompt_ex1 = ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input do usuário
em exatamente um destino:
- rag: perguntas sobre documentos, manuais e contratos do domínio
- calculadora: cálculos matemáticos, conversões numéricas e porcentagens
- conversa: saudações, perguntas gerais e qualquer outra coisa

Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
])
chain_ex1 = prompt_ex1 | llm.with_structured_output(RotaEx1)

for pergunta in ["Qual a cláusula 5 do contrato?", "Quanto é 450 × 0.88?", "Oi, tudo bem?"]:
    rota = chain_ex1.invoke({"input": pergunta})
    print(f"{pergunta[:35]:35s} → {rota.destino}")


### Exercício 2 — Quarta rota no Router Chain

**O que a solução demonstra:** a rota "resumo" preenchida em todos os pontos do andaime — `Literal`, descrição no prompt, handler e registro em `HANDLERS_EX` — com o despacho via `dict.get` achando a rota nova sozinho.

**Pontos a destacar em sala:**
- A função `rotear_ex` nem precisa mudar: o fallback do `dict.get` despacha o destino novo sozinho — a vantagem de despachar por tabela em vez de `if/elif`.
- O ponto delicado é a descrição: "resumo" precisa de exemplos que o separem de `rag` (pergunta sobre o documento) e de `conversa` (saudação) — pedidos explícitos de síntese.


In [ ]:
# Solução — quarta rota "resumo" no Router Chain (Exercício 2)
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

class RotaEx2(BaseModel):
    destino: Literal["rag", "calculadora", "conversa", "resumo"]

prompt_ex2 = ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input do usuário
em exatamente um destino:
- rag: perguntas sobre documentos do domínio
- calculadora: cálculos e conversões numéricas
- conversa: saudações e qualquer outra coisa
- resumo: pedidos de síntese — "resuma", "em poucas linhas", "o principal"

Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
])
chain_ex2 = prompt_ex2 | llm.with_structured_output(RotaEx2)

handler_resumo = (
    ChatPromptTemplate.from_messages([
        ("system", "Você resume documentos do domínio em no máximo 3 linhas, "
                   "citando apenas o que está no texto recebido."),
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

HANDLERS_EX = {
    "rag":         RunnableLambda(lambda x: "[RAG] resposta com o contexto do CKP02"),
    "calculadora": RunnableLambda(lambda x: "[CALC] resultado da expressão"),
    "conversa":    RunnableLambda(lambda x: "[CHAT] resposta amigável"),
    "resumo":      handler_resumo,
}

def rotear_ex(dados: dict) -> str:
    rota = chain_ex2.invoke({"input": dados["input"]})
    print(f"[ROUTER] {rota.destino}")
    return HANDLERS_EX.get(rota.destino, HANDLERS_EX["conversa"]).invoke(dados)

print(rotear_ex({"input": "Resuma a cláusula 5 do contrato em 2 linhas"}))


### Exercício 3 — Teste de fronteira com 3 inputs

**O que a solução demonstra:** o andaime do Exercício 3 preenchido — 3 inputs (um por rota) + 2 casos de fronteira prontos, cada um rodado 2 vezes com `temperature=0`, com detector de oscilação e a regra de desempate escrita.

**Pontos a destacar em sala:**
- "QUANTO É 450+550?" em maiúsculas pode escapar para `conversa`: a correção é instruir "ignore a capitalização" no prompt.
- O híbrido (cálculo sobre um documento) é o mais instável: oscila entre `calculadora` e `rag` mesmo com `temperature=0` — a correção não está no schema, é a regra de desempate no final da célula.
- Provocar a turma: trocar a descrição de `rag` para incluir "conceitos do domínio" e rodar de novo — a rota do 1º input muda.


In [ ]:
# Solução — teste de fronteira do classificador (Exercício 3)
from pydantic import BaseModel
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class RotaEx3(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]

clf_ex3 = (ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input em exatamente um destino:
- rag: perguntas sobre documentos do domínio
- calculadora: cálculos e conversões numéricas
- conversa: saudações e perguntas gerais
Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
]) | llm.with_structured_output(RotaEx3))

TESTES_EX = [
    ("rag", "Qual a cláusula 5 do contrato?"),
    ("calculadora", "Quanto é 450 × 0.88?"),
    ("conversa", "Oi, tudo bem?"),
    ("calculadora", "QUANTO É 450+550?"),   # caso de fronteira — pode oscilar
    ("rag", "Quanto é 15% do valor da multa prevista no contrato?"),  # híbrido
]
for esperado, pergunta in TESTES_EX:
    saidas = [clf_ex3.invoke({"input": pergunta}).destino for _ in range(2)]
    oscilou = " · oscilou" if len(set(saidas)) > 1 else ""
    ok = "✅" if saidas[0] == esperado else "❌"
    print(f"esperado={esperado:12s} obtido={saidas[0]:12s} {ok}{oscilou} | {pergunta[:42]}")

regra_hibrido = "se o input mistura cálculo e documento do domínio, prefira rag e deixe o handler resolver"
print("\nRegra de desempate:", regra_hibrido)


### Exercício 4 — Despacho declarativo — do dict.get ao mapa de rotas

**O que a solução demonstra:** o andaime do Exercício 4 preenchido — mapa de despacho completo, função lendo o campo certo do estado e a simulação rodando sem LLM — e a ponte para o `add_conditional_edges` da Aula 13.

**Pontos a destacar em sala:**
- No grafo, o despacho é declarado na estrutura: `add_conditional_edges(no_origem, fn_rota, mapa)` — o mapa no print é literalmente o 3º argumento.
- Com um 4º destino: aqui, uma linha no mapa; no `rotear()` da Aula 12, mais um `if` — com 6+ rotas e condições compostas, o emaranhado de `if`s mostra por que o grafo declarativo passa a valer a pena.


In [ ]:
# Solução — despacho declarativo: do dict.get ao mapa de rotas (Exercício 4)
# 1. Mapa completo: retorno da fn_rota → nome do nó destino
MAPA_ROTAS = {
    "rag":         "responder_rag",
    "calculadora": "responder_calc",
    "conversa":    "responder_chat",
}

# 2. A função lê o campo do estado escrito pelo node classificador
def decidir_rota_ex(estado: dict) -> str:
    return estado["rota"]

# 3. Estado de exemplo preenchido (input + decisão do classificador)
estado_sim = {"input": "Quanto é 12 × 0.9?", "rota": "calculadora"}

proximo_no = MAPA_ROTAS.get(decidir_rota_ex(estado_sim), "__end__")
print(f"fn_rota(estado) → {decidir_rota_ex(estado_sim)}")
print(f"Próximo nó: {proximo_no}")
print("Na Aula 13, este mapa vira:")
print('  builder.add_conditional_edges("classificar", decidir_rota_ex,', MAPA_ROTAS, ')')


## 📚 Referências da aula

- Docs LangChain — RunnableLambda e routing patterns. Como construir chains com branching usando RunnableLambda e with_structured_output. python.langchain.com/docs/how_to/routing
- Docs LangGraph — Conceitos de StateGraph, nodes e edges. A leitura recomendada antes da Aula 13. langchain-ai.github.io/langgraph/concepts/low_level
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Seção sobre orchestrators e subagents — a motivação arquitetural para grafos de estado. anthropic.com/engineering/building-effective-agents
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolução de problemas como busca: a teoria por trás de grafos de estado em IA, base conceitual do LangGraph.
- Livro Polzer, D. — RAG with Python Cookbook. O'Reilly, 2026. O custo de latência de um router baseado em LLM (500ms-2s) e a alternativa de classificador leve sobre embeddings — a fundamentação por trás do classificador de intenção desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 3 — Routing: as três formas de implementar roteamento (regras, classificador de ML, LLM) e o trade-off de custo/latência por trás do Router Chain desta aula.

---

**Próxima Aula — Aula 13** — LangGraph — StateGraph, conditional edges e HITL
  
O diagrama desta aula vira código. Estado TypedDict, add_node(), add_conditional_edges(), MemorySaver, interrupt_before.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*